In [ ]:
# =========================
# Step 0: Install packages
# =========================
!pip -q install pingouin openpyxl

# =========================
# Step 1: Import libraries
# =========================
from google.colab import drive, files
import pandas as pd
import numpy as np
import pingouin as pg
from scipy import stats
import matplotlib.pyplot as plt
import os

# =========================
# Step 2: Load Google Drive
# =========================
drive.mount('/content/drive')

# =========================
# Step 2: Set Excel file path
# Please ensure that c.xlsx is located in the root directory of My Drive
# =========================
file_path = '/content/drive/My Drive/inter_rater_raw_measurements.xlsx'

# If you are unsure whether the file exists, you can list the contents of My Drive first
print("=== Files in My Drive ===")
print(os.listdir('/content/drive/My Drive')[:20])

# =========================
# Step 3: Read Excel file
# =========================
df = pd.read_excel(file_path, engine='openpyxl')

print("\n=== First 5 rows of raw data ===")
print(df.head())

print("\n=== Original column names ===")
print(df.columns.tolist())

print("\n=== Data shape ===")
print(df.shape)

# =========================
# Step 4: Rename columns
# Based on column positions:
# A Chart No.
# B Level
# C,D Depth
# E,F Cord width
# G,H Disc AP
# I,J Disc width
# K,L Kambin
# =========================
if df.shape[1] < 12:
    raise ValueError(f"Insufficient number of columns. At least 12 required, but got {df.shape[1]}.")

df2 = df.iloc[:, :12].copy()
df2.columns = [
    "ChartNo",
    "Level",
    "Depth_R1", "Depth_R2",
    "CordWidth_R1", "CordWidth_R2",
    "DiscAP_R1", "DiscAP_R2",
    "DiscWidth_R1", "DiscWidth_R2",
    "Kambin_R1", "Kambin_R2"
]

print("\n=== Renamed columns ===")
print(df2.columns.tolist())

# =========================
# Step 5: Convert to numeric
# =========================
measure_pairs = {
    "Depth": ("Depth_R1", "Depth_R2"),
    "CordWidth": ("CordWidth_R1", "CordWidth_R2"),
    "DiscAP": ("DiscAP_R1", "DiscAP_R2"),
    "DiscWidth": ("DiscWidth_R1", "DiscWidth_R2"),
    "Kambin": ("Kambin_R1", "Kambin_R2"),
}

for measure, (c1, c2) in measure_pairs.items():
    df2[c1] = pd.to_numeric(df2[c1], errors="coerce")
    df2[c2] = pd.to_numeric(df2[c2], errors="coerce")

print("\n=== First 5 rows after numeric conversion ===")
print(df2.head())

# =========================
# Step 6: Define analysis function
# =========================
def analyze_two_raters(df_in, measure_name, col_r1, col_r2):
    tmp = df_in[["ChartNo", "Level", col_r1, col_r2]].dropna().copy()
    tmp = tmp.rename(columns={col_r1: "R1", col_r2: "R2"})

    n = len(tmp)
    if n < 3:
        return pd.DataFrame([{
            "Measure": measure_name,
            "N": n,
            "Note": "Not enough complete pairs"
        }]), None

    # Difference
    diff = tmp["R1"] - tmp["R2"]
    mean_diff = diff.mean()
    sd_diff = diff.std(ddof=1)
    mae = np.mean(np.abs(diff))

    # Normality of differences
    try:
        shapiro_p = stats.shapiro(diff).pvalue
    except:
        shapiro_p = np.nan

    # Systematic difference / bias
    if pd.notna(shapiro_p) and shapiro_p >= 0.05:
        test_name = "Paired t-test"
        stat, p_value = stats.ttest_rel(tmp["R1"], tmp["R2"], nan_policy="omit")
    else:
        test_name = "Wilcoxon signed-rank"
        try:
            stat, p_value = stats.wilcoxon(tmp["R1"], tmp["R2"])
        except:
            stat, p_value = np.nan, np.nan

    # Correlation
    try:
        pearson_r, pearson_p = stats.pearsonr(tmp["R1"], tmp["R2"])
    except:
        pearson_r, pearson_p = np.nan, np.nan

    # ICC long format
    long_df = pd.DataFrame({
        "targets": np.repeat(tmp["ChartNo"].astype(str).values, 2),
        "raters": ["R1", "R2"] * n,
        "scores": np.ravel(np.column_stack([tmp["R1"].values, tmp["R2"].values]))
    })

    icc_tbl = pg.intraclass_corr(
        data=long_df,
        targets="targets",
        raters="raters",
        ratings="scores"
    )

    print(f"\n=== ICC table for {measure_name} ===")
    print(icc_tbl)
    print("ICC columns:", icc_tbl.columns.tolist())

    # Automatically find CI column
    ci_col_candidates = [c for c in icc_tbl.columns if "CI" in str(c)]
    ci_col = ci_col_candidates[0] if len(ci_col_candidates) > 0 else None

    icc21 = icc_tbl.loc[icc_tbl["Type"] == "ICC2"].copy()
    icc2k = icc_tbl.loc[icc_tbl["Type"] == "ICC2k"].copy()

    if not icc21.empty:
        icc21_val = icc21["ICC"].values[0]
        icc21_p = icc21["pval"].values[0] if "pval" in icc21.columns else np.nan
        icc21_ci = icc21[ci_col].values[0] if ci_col is not None else None
    else:
        icc21_val, icc21_ci, icc21_p = np.nan, None, np.nan

    if not icc2k.empty:
        icc2k_val = icc2k["ICC"].values[0]
        icc2k_p = icc2k["pval"].values[0] if "pval" in icc2k.columns else np.nan
        icc2k_ci = icc2k[ci_col].values[0] if ci_col is not None else None
    else:
        icc2k_val, icc2k_ci, icc2k_p = np.nan, None, np.nan

    loa_lower = mean_diff - 1.96 * sd_diff
    loa_upper = mean_diff + 1.96 * sd_diff

    out = pd.DataFrame([{
        "Measure": measure_name,
        "N": n,
        "R1_mean": tmp["R1"].mean(),
        "R2_mean": tmp["R2"].mean(),
        "Mean_diff_R1_minus_R2": mean_diff,
        "SD_diff": sd_diff,
        "MAE": mae,
        "Normality_p_of_diff": shapiro_p,
        "Bias_test": test_name,
        "Bias_test_stat": stat,
        "Bias_test_p": p_value,
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "ICC2_1": icc21_val,
        "ICC2_1_p": icc21_p,
        "ICC2_1_CI": str(icc21_ci),
        "ICC2_k": icc2k_val,
        "ICC2_k_p": icc2k_p,
        "ICC2_k_CI": str(icc2k_ci),
        "BA_mean_diff": mean_diff,
        "BA_lower_LOA": loa_lower,
        "BA_upper_LOA": loa_upper
    }])

    return out, tmp

# =========================
# Step 7: Run all parameters
# =========================
results = []
paired_data_store = {}

for measure, (c1, c2) in measure_pairs.items():
    res, tmp_used = analyze_two_raters(df2, measure, c1, c2)
    results.append(res)
    paired_data_store[measure] = tmp_used

summary_df = pd.concat(results, ignore_index=True)

print("\n=== Inter-rater summary ===")
display(summary_df)

# =========================
# Step 8: Save as Excel
# =========================
output_path = '/content/drive/My Drive/inter_rater_summary.xlsx'
summary_df.to_excel(output_path, index=False)

print("\nResults saved to:", output_path)

# Also download to the local machine
files.download(output_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Files in My Drive ===
['IMG_1337.JPG', 'IMG_1338.JPG', 'IMG_1339.JPG', 'IMG_1340.JPG', 'IMG_1341.JPG', 'IMG_1342.JPG', 'IMG_1343.JPG', 'IMG_1346.JPG', 'IMG_1347.jpg', 'IMG_1348.heic', 'IMG_1349.heic', 'IMG_1350 (1).HEIC', 'IMG_1351.jpg', 'IMG_1353.HEIC', 'IMG_1350.HEIC', 'IMG_1352.HEIC', 'IMG_1354.HEIC', 'IMG_1356.heic', 'IMG_1357 (1).jpg', 'IMG_1357.jpg']

=== 原始資料前5列 ===
   Chart No. Level  Depth  Unnamed: 3  Cord width  Unnamed: 5  Disc AP  \
0   29100687   L12   59.4        58.6        21.1        19.3     40.6   
1     289873   L23   63.3        62.3        22.7        22.1     35.2   
2    1191384   L34   59.8        59.0        10.2        10.0     42.2   
3    2057110   L45   63.3        65.0         9.8        10.2     44.2   
4    2776087   L51   77.5        76.2        16.9        15.5     35.0   

   Unnamed: 7  Disc width  Unnamed: 9  Kambin 

,Measure,N,R1_mean,R2_mean,Mean_diff_R1_minus_R2,SD_diff,MAE,Normality_p_of_diff,Bias_test,Bias_test_stat,...,Pearson_p,ICC2_1,ICC2_1_p,ICC2_1_CI,ICC2_k,ICC2_k_p,ICC2_k_CI,BA_mean_diff,BA_lower_LOA,BA_upper_LOA
0,Depth,30,65.856667,65.323333,0.533333,0.849882,0.833333,0.348860,Paired t-test,3.437169,...,3.765106e-30,0.993833,1.761381e-31,[0.98 1. ],0.996907,1.761381e-31,[0.99 1. ],0.533333,-1.132435,2.199101
1,CordWidth,30,17.643333,17.196667,0.446667,0.794261,0.746667,0.572316,Paired t-test,3.080214,...,4.455578e-22,0.976474,1.075200e-22,[0.93 0.99],0.988097,1.075200e-22,[0.97 1. ],0.446667,-1.110085,2.003418
2,DiscAP,30,38.670000,38.253333,0.416667,0.710553,0.623333,0.503492,Paired t-test,3.211834,...,1.942409e-23,0.981171,3.172286e-24,[0.94 0.99],0.990496,3.172286e-24,[0.97 1. ],0.416667,-0.976017,1.809350
3,DiscWidth,30,54.936667,54.510000,0.426667,0.817453,0.780000,0.033517,Wilcoxon signed-rank,94.000000,...,3.448869e-25,0.987275,2.428985e-26,[0.97 0.99],0.993597,2.428985e-26,[0.98 1. ],0.426667,-1.175542,2.028875
4,Kambin,30,11.856667,11.763333,0.093333,1.030545,0.880000,0.335037,Paired t-test,0.496056,...,2.488309e-23,0.968335,3.586895e-19,[0.93 0.98],0.983913,3.586895e-19,[0.97 0.99],0.093333,-1.926535,2.113202



結果已儲存到： /content/drive/My Drive/inter_rater_summary.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>